# Parameter tester

Quick, no-subsampling playground: **load** a dataset, optionally **add** signature targets / cancer factors, then **fit once** and look at the betas. Nothing is cached.

## 1. Load

In [ ]:
import sys
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from metab_processing.LinearRegression import signature_helpers as sh
from metab_processing.LinearRegression.signature_configs import CONFIGS
from metab_processing.LinearRegression.least_squares import fit_gene_betas, rank_coefficients

In [ ]:
# Pick a dataset (any key in CONFIGS: 'cervical' | 'breast' | 'colon').
cfg = CONFIGS['cervical']
adata = sc.read_h5ad(cfg.adata_path)
var = set(adata.var_names)
print(adata.n_obs, 'cells |', len(adata.uns.get('x_genes', [])), 'built genes')

# Cell selection + fit settings default to the dataset's config; override inline to test.
ANNOT_COL = cfg.annot_col or cfg.cell_type_col
CELL_TYPE = cfg.cell_type or (cfg.t_cell_types[0] if cfg.t_cell_types else None)
METHOD, PENALTY, L1_RATIO, STANDARDIZE = cfg.method, cfg.penalty, cfg.l1_ratio, cfg.standardize
print('annot_col =', ANNOT_COL, '| cell_type =', CELL_TYPE,
      '| method =', METHOD, '| penalty =', PENALTY)

## 2. (optional) add signature targets / cancer factors

Uncomment what you want. Leave both commented to fit a plain gene.

In [ ]:
library = None  # lazy: fetched once below if any optional block needs it

# --- OPTION A: add cancer gene-set signatures as spatial-neighborhood FACTORS (cancers) ---
# library = library or sh.load_msigdb(cfg.dbver, cfg.categories)
# cancer_sets = {n: [g for g in library.get(n, []) if g in var] for n in cfg.cancer_factors}
# cancer_sets = {k: v for k, v in cancer_sets.items() if v}
# sh.add_cancer_factors(adata, cancer_sets, annot_col=cfg.cell_type_col, add_proximity=True)

# --- OPTION B: build a signature pseudo-gene TARGET (instead of a raw gene) ---
# library = library or sh.load_msigdb(cfg.dbver, cfg.categories)
# SIG = 'HALLMARK_IL2_STAT5_SIGNALING'
# genes = [g for g in library.get(SIG, []) if g in var]
# adata = sh.add_gene_signature(adata, SIG, positive=genes, keep_when_overlap='factor', verbose=True)
# TARGET = SIG

# Default TARGET: a raw built gene. Edit to any gene in adata.uns['x_genes'].
TARGET = cfg.single_gene or adata.uns['x_genes'][0]
print('TARGET =', TARGET)

## 3. Fit once (no subsampling) and view the betas

In [ ]:
betas = fit_gene_betas(adata, genes=[TARGET], annot_col=ANNOT_COL, annot_value=CELL_TYPE,
                       metabolites=[], custom='all', method=METHOD, penalty=PENALTY,
                       l1_ratio=L1_RATIO, standardize=STANDARDIZE)
rank_coefficients(betas).head(60)